# Trabalho Prático de Mineração de Dados — Fase 2
## Utilização de LLMs e Análise Comparativa com Diretrizes de IA Responsável

**Disciplina:** Mineração de Dados  
**Grupo:** João Marcos Oliveira Neves (2024008644), Júlia Borba Fonseca de Souza (2023117580), Laura Martins Froede (2023087877), Matheus Soares dos Santos de Freitas (2024080043)
**Dataset:** Adult Census Income  
**Tarefa de mineração:** Classificação
**Data da entrega:** 14/06/2026

---

## Objetivo deste notebook

Este notebook documenta a Fase 2 do Trabalho Prático. O objetivo é usar uma ou mais LLMs como apoio à construção da solução de mineração de dados e comparar duas trilhas de interação:

- **Trilha A — baseline:** interação com a LLM sem explicitar diretrizes de IA responsável;
- **Trilha B — guiada:** interação com a LLM explicitando as diretrizes de IA responsável escolhidas pelo grupo.

O foco não é apenas mostrar prompts e respostas. O grupo deve analisar criticamente se a explicitação das diretrizes levou a diferenças concretas na solução proposta, por exemplo em:

- escolha de algoritmos;
- tratamento dos dados;
- seleção de atributos;
- definição de parâmetros;
- métricas de avaliação;
- interpretabilidade;
- privacidade;
- análise de viés;
- custo computacional;
- qualidade da documentação.

> **Importante:** este notebook é um modelo de estrutura. Ele contém exemplos, placeholders e código genérico. O grupo deve substituir os campos indicados por informações reais do seu dataset, das interações com LLMs e dos experimentos efetivamente realizados.


# Como usar este notebook

Este notebook foi organizado seguindo a estrutura esperada para a Fase 2:

1. **Business Understanding**
2. **Data Understanding & Data Preparation**
3. **Modeling**
4. **Evaluation**
5. **Checklist final**

Ao longo do notebook, os trechos marcados entre colchetes, como `[INSERIR LINK DA CONVERSA]`, devem ser preenchidos pelo grupo.

## Diferença entre exemplo e entrega real

- Células de **exemplo** mostram como organizar a entrega.
- Células com **placeholders** indicam pontos que devem ser preenchidos.
- Células de **código** são genéricas e podem precisar de adaptação ao dataset.
- Nenhum resultado experimental deve ser inventado.
- Toda conclusão empírica deve estar apoiada em uma tabela, gráfico ou execução de código.


# 0. Controle da entrega e rastreabilidade

Esta seção registra informações mínimas para tornar o trabalho rastreável e auditável.

| Item | Valor |
|---|---|
| Nome do dataset | Adult Census Income |
| Link público do dataset | https://www.kaggle.com/datasets/uciml/adult-census-income |
| Número de linhas | 32.561 |
| Número de colunas | 15 |
| Tarefa de mineração | Classificação |
| LLM usada na Trilha A | Gemini Pro |
| LLM usada na Trilha B | Gemini Pro |
| Link da conversa — Trilha A | https://gemini.google.com/share/68fcb4667de8 |
| Link da conversa — Trilha B | https://gemini.google.com/share/abcee9d7e641 |

## Diretrizes de IA responsável escolhidas

| Diretriz | Justificativa da escolha | Como será explicitada na Trilha B |
|---|---|---|
| Justiça e Mitigação de Viés | O dataset Adult Census Income possui um desbalanceamento histórico e estrutural severo, onde subgrupos de gênero e raça apresentam proporções de alta renda significativamente menores devido ao contexto social do Censo de 1994. A escolha desta diretriz é indispensável para auditar o comportamento do algoritmo e garantir que o classificador não converta essas desigualdades e correlações históricas em regras automatizadas de exclusão, operando de forma discriminatória direta ou indiretamente por meio de variáveis proxy (como ocupação ou relacionamento). | "Atue como um Auditor Independente de Viés Algorítmico. Analise as taxas de falsos negativos e o Impacto Díspar entre os subgrupos de gênero e raça para o modelo XGBoost treinado. Avalie se o algoritmo está perpetuando discriminação de forma direta ou por meio de variáveis proxy e proponha uma estratégia de pós-processamento (como ajuste de limiar de decisão) para equalizar as oportunidades." |
| Explicabilidade | O modelo selecionado (XGBoost) é um algoritmo de conjunto complexo baseado em árvores de decisão impulsionadas que, por sua natureza matemática não linear, opera originalmente como uma "caixa-preta" abstrata para humanos. A escolha desta diretriz justifica-se pela necessidade ética e técnica de abrir essa estrutura e mapear com precisão os fatores reais de negócio (como educação e ganho de capital) que fundamentam cada predição individual e global, garantindo transparência, auditabilidade e o direito à explicação. | "Atue como um Especialista em Inteligência Artificial Explicável (XAI). Com base nos valores SHAP globais e nos gráficos de dependência parcial gerados para o XGBoost, construa uma justificativa textual clara e acessível ao público geral explicando quais fatores profissionais (ex: nível educacional) ou de capital foram determinantes para a classificação de renda efetuada." |

> **Propósito pedagógico:** esta seção evita que a análise fique solta ou apenas narrativa. O avaliador precisa conseguir identificar o que foi perguntado à LLM, quando, em qual modelo e com qual objetivo.


In [1]:
# Configurações iniciais do notebook
# Esta célula centraliza imports e parâmetros gerais.
# O grupo deve adaptar os caminhos, nomes de colunas e tipo de tarefa.

import os
import time
import json
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

# Caminho para o dataset.
# Exemplos:
# DATA_PATH = "dados/dataset.csv"
# DATA_PATH = "/content/dataset.csv"
DATA_PATH = "https://raw.githubusercontent.com/Doctor-Math/TPs-Mineracao-de-Dados/VariableMath/data/data_tp3/adult.csv"

# Tipo de tarefa do TP.
# Opções sugeridas:
# - "padroes_frequentes"
# - "agrupamento"
# - "classificacao"
TASK_TYPE = "classificacao"

# Coluna-alvo, se aplicável.
# Para padrões frequentes ou agrupamento, normalmente pode ser None.
TARGET_COLUMN = None  # Exemplo: "classe"

# Coluna ou conjunto de colunas usadas para gerar transações, se o TP for de padrões frequentes.
TRANSACTION_COLUMN = "[INSERIR_COLUNA_DE_TRANSACOES]"

print("Configuração carregada.")
print(f"Tarefa definida: {TASK_TYPE}")


Configuração carregada.
Tarefa definida: classificacao


# 1. Business Understanding

Nesta seção, o grupo deve explicar o problema, o contexto dos dados, a relevância da tarefa e a hipótese experimental sobre o uso das diretrizes de IA responsável.

A seção deve responder:

1. Qual problema será investigado?
2. Por que o dataset é relevante?
3. Qual tarefa de mineração de dados será executada?
4. Qual é o valor esperado da análise?
5. Quais diretrizes de IA responsável foram escolhidas?
6. Por que essas diretrizes são pertinentes ao problema?
7. O que se espera que mude quando essas diretrizes são explicitadas à LLM?


## 1.1 Descrição do problema de negócio
- **Contexto:** Instituições governamentais, organizações do terceiro setor e departamentos de planejamento estratégico de empresas privadas frequentemente necessitam segmentar e compreender o perfil socioeconômico de populações para formular políticas públicas de redistribuição de renda, desenhar programas de assistência social ou direcionar ofertas de serviços essenciais. O mapeamento preciso da capacidade financeira individual, historicamente balizado por censos demográficos nacionais (como o Censo de 1994), serve como a fundação estrutural para estimar o bem-estar econômico e o potencial de desenvolvimento de uma sociedade.
- **Problema:** A coleta direta e exaustiva de dados de renda declarada é uma operação complexa, onerosa e sujeita a altas taxas de omissão ou subdeclaração por parte dos cidadãos. Diante disso, o problema central consiste em predizer de forma automatizada se a renda anual de um indivíduo supera ou não o limiar de USD 50.000 utilizando variáveis demográficas, educacionais e ocupacionais como preditores. O desafio analítico reside no fato de que os dados históricos herdam desigualdades estruturais de gênero e raça da época de sua coleta. Utilizar um classificador de alta capacidade (como o XGBoost) sem salvaguardas éticas criará um sistema que perpetua e automatiza essas discriminações do passado, mascarando preconceitos sob o pretexto de "otimização matemática".
- **Possíveis interessados:**
 - *Gestores de Políticas Públicas e Ministérios Sociais:* Interessados em identificar com precisão bolsas de pobreza ou regiões com maior concentração de indivíduos de baixa renda para alocação eficiente de recursos e subsídios, exigindo que os critérios de elegibilidade sejam transparentes e justos.
 - *Analistas de Crédito e Risco de Instituições Financeiras:* Que utilizam a estimativa de renda para avaliar a capacidade de endividamento e mitigar o risco de inadimplência na concessão de microcrédito.
 - *Comitês de Auditoria de IA e Ombudsman Éticos:* Profissionais responsáveis por garantir que as ferramentas preditivas da instituição estejam em conformidade com regulações de equidade, evitando processos jurídicos por discriminação algorítmica indireta.
- **Decisões apoiadas pela análise:**
 - *Concessão Justa de Benefícios ou Crédito:* Definição automatizada ou semiautomatizada de quais cidadãos preenchem os requisitos de faixa de renda para acessar incentivos fiscais, bolsas de estudo ou linhas de financiamento habitacional.
 - *Ajuste de Limiares de Decisão (Thresholds) por Equidade:* Decisão técnica e operacional de calibrar o ponto de corte do classificador XGBoost para equalizar as taxas de falsos negativos entre diferentes gêneros e raças, aceitando uma pequena variação na precisão global para garantir o cumprimento do princípio de Impacto Díspar Neutro.
 - *Intervenções de Capacitação Profissional:* Identificação de quais setores de ocupação ou níveis educacionais funcionam como os principais alavancadores de renda para guiar investimentos em escolas técnicas e programas de recolocação.
- **Limitações iniciais conhecidas:**
 - *Anacronismo dos Dados Históricos:* O dataset reflete a realidade socioeconômica dos Estados Unidos no ano de 1994. Portanto, as relações de valor e as distribuições de renda encontradas estão desatualizadas em relação à inflação e às dinâmicas do mercado de trabalho contemporâneo, limitando a validação prática a um cenário descritivo de simulação.
 - *Falta de Contexto Macroeconômico Atualizado:* A base carece de indicadores modernos importantes, como score de crédito, custo de vida regional, patrimônio acumulado ou arranjo familiar detalhado, restringindo o poder de explicação do modelo aos atributos contidos no censo.
 - *Desbalanceamento Crítico da Variável-Alvo:* A forte assimetria na distribuição da classe de renda (onde cerca de 76% da base ganha $\le$ 50K) impõe uma barreira severa ao treinamento do XGBoost, que pode inclinar suas fronteiras de decisão para ignorar a classe minoritária de alta renda, exigindo métricas de avaliação que penalizem o erro de forma assimétrica.

> Este texto deve ser escrito pelo grupo. Não copie a descrição do dataset sem contextualizar o problema de mineração de dados.


## 1.2 Objetivo do dataset, origem e características gerais

### Objetivo do dataset

O propósito central do dataset *Adult Census Income* é servir como uma base de referência para o desenvolvimento e avaliação de modelos de classificação supervisionada binária. O objetivo técnico consiste em predizer se a renda anual de um indivíduo excede ou não o patamar de 50 mil dólares (>50K ou <=50K) com base em um conjunto misto de atributos demográficos, educacionais e laborais. Sob a perspectiva de IA Responsável, o objetivo do dataset expande-se para atuar como um ambiente de testes (*benchmark*) para auditoria de algoritmos, identificação de preconceitos históricos ocultos e aplicação de técnicas de mitigação de impacto díspar e engenharia de explicabilidade pós-hoc.

### Origem dos dados

Os dados foram originalmente extraídos dos registros do escritório do Censo dos Estados Unidos (*United States Census Bureau*) relativos ao ano de 1994. A extração e a formatação inicial da base foram realizadas por Ronny Kohavi e Barry Becker, sendo posteriormente doadas ao Repositório de Aprendizado de Máquina da Universidade de Califórnia em Irvine (UCI Machine Learning Repository). O conjunto de dados tornou-se mundialmente difundido e está publicamente hospedado na plataforma Kaggle.

Link público: https://www.kaggle.com/datasets/uciml/adult-census-income](https://www.kaggle.com/datasets/uciml/adult-census-income

### Características do dataset

Abaixo estão detalhados os principais atributos que compõem a base de dados, mapeando sua natureza e papel na construção do classificador:

| Coluna | Tipo esperado | Descrição | Relevância para a tarefa |
| --- | --- | --- | --- |
| `age` | Numérico (`int64`) | Idade do indivíduo em anos. | Altamente relevante por capturar o efeito do tempo de carreira e maturidade profissional no potencial de ganho. |
| `workclass` | Categórico (`object`) | Setor de ocupação trabalhista (ex: Private, Self-emp, Federal-gov, etc). | Crucial para entender a distribuição salarial entre a iniciativa privada, funcionalismo público e autônomos. |
| `education` | Categórico (`object`) | O mais alto nível de escolaridade formal alcançado pelo indivíduo. | Fornece o contexto nominal da formação, sendo uma variável de forte apelo descritivo de negócio. |
| `education-num` | Numérico (`int64`) | Nível educacional codificado estritamente pelo total de anos de estudo. | Altamente relevante por ser uma representação contínua da educação, ideal para o particionamento do XGBoost sem explodir a dimensionalidade. |
| `marital-status` | Categórico (`object`) | Estado civil do indivíduo (ex: Married-civ-spouse, Divorced, Never-married). | Relevante para o modelo pois, historicamente, o arranjo civil está correlacionado à estabilidade econômica e tributária familiar. |
| `occupation` | Categórico (`object`) | A ocupação profissional técnica exata do indivíduo (ex: Exec-managerial, Sales, Craft-repair). | Atributo de altíssimo poder preditivo que mapeia o teto de remuneração de cada setor de mercado. |
| `relationship` | Categórico (`object`) | Papel ou posição do indivíduo dentro do núcleo familiar (ex: Husband, Wife, Own-child). | Atua como uma forte variável preditiva, mas exige cuidado ético por carregar altos índices de colinearidade com gênero. |
| `race` | Categórico (`object`) | Etnia/raça autodeclarada do indivíduo (White, Black, Asian-Pac-Islander, etc). | **Atributo Sensível Crítico:** Deve ser auditado de forma estrita para evitar discriminação direta e avaliar o impacto díspar. |
| `sex` | Categórico (`object`) | Gênero biológico registrado (Male, Female). | **Atributo Sensível Crítico:** Utilizado diretamente na linha de frente para monitorar a equidade algorítmica e taxas de falsos negativos. |
| `capital-gain` | Numérico (`int64`) | Ganhos de capital financeiro obtidos por investimentos externos ao salário. | Variável monetária de cauda longa, de altíssima relevância para isolar indivíduos de renda muito elevada. |
| `capital-loss` | Numérico (`int64`) | Perdas de capital financeiro registradas no período. | Indica o envolvimento em investimentos de risco, servindo como proxy de movimentação patrimonial. |
| `hours-per-week` | Numérico (`int64`) | Quantidade de horas trabalhadas por semana. | Mede diretamente a intensidade da jornada laboral, correlacionando-se ao volume financeiro bruto recebido. |
| `native-country` | Categórico (`object`) | País de origem de nascimento do indivíduo. | Variável de alta cardinalidade utilizada para mapear o impacto geográfico e imigratório na renda. |
| `income` | Categórico (`object`) | **Variável-Alvo:** Rótulo indicando a faixa de renda anual (\texttt{<=50K} ou \texttt{>50K}). | É o alvo da classificação supervisionada. Determina o sucesso do aprendizado do XGBoost através de sua fronteira de decisão. |

*(Nota: O atributo original `fnlwgt` foi omitido desta tabela conceitual por se tratar de um peso amostral sociodemográfico populacional calculado pelo próprio censo da época, não possuindo correlação comportamental útil para o aprendizado do algoritmo.)*

### Relação com o problema de negócio

O dataset *Adult Census Income* é perfeitamente adequado para o problema de negócio escolhido devido à sua natureza multidimensional, que espelha os desafios analíticos complexos enfrentados por instituições ao tomarem decisões de concessão de crédito, incentivos ou benefícios.

Primeiramente, a base fornece o ecossistema ideal para testar a robustez do **XGBoost**, dada a presença massiva de atributos categóricos nominais cruzados com atributos numéricos de cauda longa e severamente assimétricos (como `capital-gain`). Em segundo lugar, o dataset resolve diretamente o problema de negócio de como gerenciar e mitigar vieses: ao disponibilizar variáveis demográficas explícitas (`sex` e `race`), ele permite quantificar de forma exata e matemática o desbalanceamento histórico nas decisões do modelo.

Por fim, a base atende à exigência de **Explicabilidade** corporativa, uma vez que o modelo final treinado precisará justificar, via valores SHAP, se a classificação de um indivíduo em uma faixa de renda menor decorreu de critérios técnicos legítimos de mercado (como escolaridade e horas trabalhadas) ou se foi contaminada de forma discriminatória indireta através das variáveis de controle.

## 1.3 Diretrizes selecionadas e hipótese experimental

### Diretrizes selecionadas

| Diretriz | Por que é pertinente ao problema? | Possível efeito esperado na solução da LLM |
| --- | --- | --- |
| **Justiça e Mitigação de Viés** | O dataset herda as disparidades socioeconômicas estruturais dos EUA de 1994, registrando proporções historicamente menores de alta renda para mulheres e minorias éticas. Como o XGBoost possui alta capacidade de memorização e mapeamento não linear, ele tende a codificar essas desigualdades como regras preditivas rígidas. A diretriz é indispensável para forçar a auditoria das taxas de erro e impedir que o algoritmo discrimine de forma direta ou por variáveis *proxy*. | Espera-se que a LLM rejeite a acurácia global isolada e exija métricas de equidade (como Impacto Díspar ou Paridade Demográfica). Espera-se também que ela recomende pipelines de preparação focados em neutralidade, como a remoção ou o tratamento estatístico de atributos de alta colinearidade com gênero e raça (ex: `relationship`), e sugira pós-processamentos para calibração de limiares (*thresholds*). |
| **Explicabilidade** | O XGBoost funciona como um comitê sequencial de árvores complexas (*ensemble*), gerando uma estrutura matemática que opera como "caixa-preta" para tomadores de decisão humanos. Como classificar a renda de um cidadão impacta diretamente sua elegibilidade a benefícios ou crédito, é imperativo abrir o modelo para garantir transparência, auditabilidade e o direito à explicação do indivíduo. | Espera-se que a LLM proponha a integração obrigatória de métodos pós-hoc de explicabilidade agnósticos ao modelo, especificamente os valores **SHAP** (SHapley Additive exPlanations). Em vez de aceitar apenas métricas globais de erro, ela deve exigir a geração de gráficos de importância de atributos locais e globais para traduzir as decisões do XGBoost em justificativas de negócio inteligíveis. |

### Hipótese experimental

> Esperamos que, ao explicitar as diretrizes de *Justiça e Mitigação de Viés* e de *Explicabilidade* nas instruções da Trilha B (guiada), a LLM proponha um pipeline de classificação com **maior rigor ético no tratamento de dados, métricas de avaliação multidimensionais focadas em equidade e a incorporação mandatória de ferramentas de auditoria pós-hoc (SHAP)**, em comparação com a solução baseline (Trilha A). Na Trilha A, sem o direcionamento explícito dessas diretrizes, antecipa-se uma abordagem puramente tecnicista e focada na otimização matemática cega do XGBoost, priorizando a performance preditiva global (foco em métricas agregadas) em detrimento da transparência e da representatividade dos subgrupos protegidos.

### Evidências que serão analisadas

Para avaliar a hipótese, compararemos as trilhas quanto a:

* **algoritmos sugeridos:** se haverá uma discussão sobre o uso de modelos intrinsecamente interpretáveis versus o XGBoost e como a complexidade do algoritmo escolhido é balanceada;
* **etapas de preparação dos dados:** se o pré-processamento inclui a engenharia de atributos neutros, tratamento de variáveis *proxy* e cuidados com o desbalanceamento sem introduzir novos vieses;
* **parâmetros recomendados:** se os hiperparâmetros sugeridos para o XGBoost incluem controles de regularização ($\mathbf{L1}$ e $\mathbf{L2}$) ou balanceamento de pesos de classe (`scale_pos_weight`) para lidar com a assimetria da base;
* **métricas propostas:** se a avaliação incorpora métricas de equidade e justiça algorítmica ou se permanece limitada a indicadores tradicionais de desempenho técnico;
* **justificativas técnicas:** o nível de profundidade conceitual ao defender as escolhas de engenharia de recursos e o ponto de corte (*threshold*) do classificador;
* **riscos apontados:** se a LLM detecta a reprodução automática de desigualdades históricas de gênero e raça contidas no censo de 1994;
* **limitações reconhecidas:** a capacidade de mapear a perda de interpretabilidade nativa do modelo e o custo computacional associado à explicabilidade pós-hoc;
* **aderência às diretrizes escolhidas:** o grau de engajamento do modelo de linguagem em atuar como um auditor crítico e transparente em vez de um mero otimizador de código.

## 1.4 Interações com LLM — Business Understanding

### Trilha A — baseline

**Modelo utilizado:** Gemini Pro
**Data de acesso:** 06/07/2026
**Link da conversa:** (https://gemini.google.com/share/8c72f57d7c9e)
**Subtarefa:** obter apoio para formular o Business Understanding.

#### Prompt principal — baseline

```text
Estou fazendo um trabalho de Mineração de Dados com o dataset Adult Census Income.
A tarefa é Classificação Supervisionada Binária utilizando o algoritmo XGBoost.
Ajude-me a formular o Business Understanding, incluindo objetivo do dataset,
origem dos dados, características gerais e relação com o problema.
```

#### Síntese da resposta da LLM

1. Contexto e Objetivo do Dataset
O dataset Adult Census Income (também conhecido como "Predict Census Income" ou simplesmente "UCI Adult Dataset") foi extraído do banco de dados do Censo de 1994 dos Estados Unidos por Ronny Kohavi e Barry Becker.
- Objetivo Primário: Determinar se a renda de uma pessoa excede o valor de US$ 50.000 por ano com base em dados demográficos e empregatícios.
- A Target: A variável alvo é binária (>50K ou ≤50K).O "Problema de Negócio"Embora a extração original visasse testar algoritmos de aprendizado de máquina, no contexto de negócios e políticas públicas, a capacidade de prever a faixa de renda de um indivíduo é crucial para:Instituições Financeiras: Avaliação de risco de crédito, concessão de empréstimos e oferta de produtos premium.Marketing Direcionado: Segmentação de clientes para produtos de alto valor.
- Governo e ONGs: Formulação de políticas socioeconômicas, alocação de recursos de assistência social e identificação de fatores que influenciam a desigualdade de renda.
2. Origem e Características Gerais dos Dados
Os dados são de natureza pública, extraídos do U.S. Census Bureau. 
O dataset possui 32.561 instâncias (no conjunto de treino padrão) e 14 atributos (recursos/features), além da variável preditora. Uma característica marcante é a mistura de tipos de dados:
- Variáveis Numéricas contínuas: Idade (age), ganho de capital (capital-gain), perda de capital (capital-loss) e horas trabalhadas por semana (hours-per-week).
- Variáveis Categóricas nominais e ordinais: Escolaridade (education, education-num), estado civil (marital-status), ocupação (occupation), relacionamento familiar (relationship), raça (race), sexo (sex) e país de origem (native-country).

Desafios Inerentes aos Dados (Data Understanding Preliminar)
- Desbalanceamento de Classes: Aproximadamente 24% das instâncias possuem renda >50K, enquanto 76% ganham ≤50K. Esse desbalanceamento deve ser levado em conta na hora de avaliar o XGBoost (usando métricas como F1-Score ou AUC-ROC em vez de apenas Acurácia).
- Dados Ausentes: Existem valores omitidos (frequentemente representados por ?) em variáveis como workclass, occupation e native-country.

3. Relação com o Problema: 
Por que usar XGBoost? A modelagem preditiva neste dataset busca mapear como características estruturais e comportamentais impactam o poder aquisitivo. O algoritmo XGBoost (Extreme Gradient Boosting) se justifica por várias razões de negócio e técnicas:
- Relações Não-Lineares e Interações: Fatores como "Idade" e "Horas Trabalhadas" não aumentam a renda de forma puramente linear. O pico de ganho geralmente ocorre na meia-idade. Modelos baseados em árvores de decisão capturam essas nuances perfeitamente.
- Tratamento de Variáveis Categóricas: O XGBoost lida muito bem com o grande volume de features categóricas deste dataset (após a devida codificação como Target Encoding ou One-Hot Encoding).
- Regularização: Como o dataset possui atributos que podem gerar forte correlação ou sobreajuste (ex: education e education-num medem conceitos muito similares), os parâmetros de regularização L1 ($\lambda$) e L2 ($\alpha$) do XGBoost ajudam a manter o modelo generalizável.

---

### Trilha B — guiada por diretrizes

**Modelo utilizado:** Gemini Pro 
**Data de acesso:** 07/06/2026 
**Link da conversa:** https://gemini.google.com/share/2f4b53d8f401
**Subtarefa:** obter apoio para formular o Business Understanding considerando as diretrizes escolhidas.

#### Prompt principal — guiado

```text
Estou fazendo um trabalho de Mineração de Dados com o dataset Adult Census Income.
A tarefa é Classificação Supervisionada Binária utilizando o algoritmo XGBoost para prever se a renda de um indivíduo excede 50K anuais.
Ajude-me a formular o Business Understanding.

Considere explicitamente as seguintes diretrizes de IA responsável:
1. Justiça e Mitigação de Viés: O censo de 1994 herda desigualdades históricas de gênero (sex) e raça (race). O modelo não pode perpetuar ou automatizar essas discriminações como regras preditivas rígidas.
2. Explicabilidade: O XGBoost é um modelo ensemble de alta complexidade matemática (caixa-preta). Precisamos auditar e explicar o peso de variáveis legítimas (educação, capital-gain) versus atributos sensíveis de forma transparente.

Além da descrição do problema, indique riscos, limitações e critérios éticos que deveriam orientar a solução de mineração de dados.
```

#### Síntese da resposta da LLM

# 1. Business Understanding (Entendimento do Negócio)

## 1.1. Contexto e Objetivo
O objetivo deste projeto é aplicar o algoritmo **XGBoost** para uma tarefa de Classificação Binária utilizando o dataset *Adult Census Income* (1994). A meta é prever se a renda anual de um indivíduo excede US$ 50K com base em atributos censitários. Contextos de negócio para este tipo de modelo incluem *credit scoring*, marketing direcionado e alocação de recursos públicos.

## 1.2. Diretrizes de IA Responsável
Como sistemas preditivos baseados em dados históricos tendem a automatizar desigualdades do passado, este projeto adota diretrizes estritas:

* **Justiça e Mitigação de Viés:** O censo de 1994 contém vieses históricos profundos em relação a gênero (`sex`) e raça (`race`). O objetivo ético é evitar que o modelo utilize essas desigualdades estruturais como regras preditivas, buscando métricas de equidade (como *Equalized Odds*).
* **Explicabilidade (Auditoria do XGBoost):** Sendo o XGBoost um modelo de alta complexidade (*caixa-preta*), a auditoria de suas decisões é obrigatória. Utilizaremos ferramentas como **SHAP values** para garantir de forma transparente que variáveis legítimas (como `education` e `capital-gain`) conduzam a predição, monitorando o impacto dos atributos sensíveis.

## 1.3. Riscos e Limitações
* **Variáveis Proxy:** Risco do algoritmo recriar vieses através de variáveis correlacionadas aos atributos sensíveis (ex: `relationship` ou `occupation` agindo como *proxies* para gênero).
* **Obsolescência dos Dados:** Dados de 1994 não refletem a economia atual. O modelo serve estritamente para propósitos de pesquisa e estudo algorítmico, não devendo ser implantado em produção sem dados contemporâneos.
* **Critério de Sucesso:** O sucesso do projeto será medido não apenas pela maximização do F1-Score ou ROC-AUC, mas pela distribuição equitativa dos erros entre diferentes grupos demográficos.

## 1.5 Análise crítica das trilhas — Business Understanding

| Critério | Trilha A — baseline | Trilha B — guiada | Diferença observada |
|---|---|---|---|
| Clareza do problema | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Relação com o dataset | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Consideração das diretrizes | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Identificação de riscos | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Qualidade da justificativa | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Erros ou omissões | [ANALISAR] | [ANALISAR] | [COMPARAR] |

### Síntese crítica

[INSERIR ANÁLISE COMPARATIVA]

A resposta guiada deve ser considerada melhor apenas se houver evidência concreta de melhoria. Exemplos de evidência:

- mencionou atributos sensíveis que a baseline ignorou;
- propôs critérios de avaliação mais adequados;
- apontou limitações metodológicas relevantes;
- sugeriu documentação mais rastreável;
- evitou recomendações inadequadas feitas na baseline.

Caso não haja diferença relevante, isso também deve ser relatado.


# 2. Data Understanding & Data Preparation

Nesta seção, o grupo deve explorar o dataset, identificar problemas de qualidade dos dados e preparar os dados para a etapa de modelagem.

Além disso, deve registrar como a LLM foi usada para apoiar:

- exploração inicial;
- identificação de problemas;
- tratamento de valores ausentes;
- transformação de atributos;
- seleção de variáveis;
- preparação específica para a tarefa de mineração;
- identificação de aspectos relacionados às diretrizes de IA responsável.


In [2]:
# Carregamento do dataset
# Esta célula deve ser adaptada de acordo com o formato do arquivo.

from pathlib import Path
import pandas as pd

def carregar_dataset(caminho: str) -> pd.DataFrame:
    """
    Carrega o dataset a partir de um caminho local ou de uma URL da internet.
    """
    # Verifica se o caminho é uma URL
    if str(caminho).startswith(("http://", "https://")):
        if caminho.endswith(".csv"):
            return pd.read_csv(caminho)
        else:
            raise ValueError("Formato de URL não tratado neste modelo. Esperado um arquivo .csv.")
            
    # Se não for URL, trata como arquivo local usando Pathlib
    caminho_path = Path(caminho)
    
    if not caminho_path.exists():
        raise FileNotFoundError(f"Arquivo local não encontrado: {caminho}")
        
    if caminho_path.suffix.lower() == ".csv":
        return pd.read_csv(caminho_path)
    
    raise ValueError("Formato não tratado neste modelo. Adapte a função carregar_dataset.")

# Substitua o seu DATA_PATH pelo link raw que geramos:
DATA_PATH = "https://raw.githubusercontent.com/Doctor-Math/TPs-Mineracao-de-Dados/VariableMath/data/data_tp3/adult.csv"

# Exemplo de uso:
df = carregar_dataset(DATA_PATH)
print("Dataset carregado com sucesso!")
print(f"Número de linhas: {df.shape[0]}")
print(f"Número de colunas: {df.shape[1]}")
display(df.head())


Dataset carregado com sucesso!
Número de linhas: 32561
Número de colunas: 15


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [3]:
# Verificação mínima das restrições do dataset
# A especificação exige dataset público, com pelo menos 1000 linhas e 4 colunas.
# O link público deve ser documentado em Markdown; aqui validamos apenas dimensões.

def verificar_restricoes_dataset(df: pd.DataFrame) -> None:
    n_linhas, n_colunas = df.shape

    print("Verificação das dimensões do dataset")
    print(f"- Linhas: {n_linhas}")
    print(f"- Colunas: {n_colunas}")

    if n_linhas < 1000:
        print("ATENÇÃO: o dataset possui menos de 1000 linhas.")
    else:
        print("OK: o dataset possui pelo menos 1000 linhas.")

    if n_colunas < 4:
        print("ATENÇÃO: o dataset possui menos de 4 colunas.")
    else:
        print("OK: o dataset possui pelo menos 4 colunas.")

# Executar após carregar o dataset:
verificar_restricoes_dataset(df)


Verificação das dimensões do dataset
- Linhas: 32561
- Colunas: 15
OK: o dataset possui pelo menos 1000 linhas.
OK: o dataset possui pelo menos 4 colunas.


In [4]:
# Exploração inicial
# Esta célula gera uma visão geral do dataset.
# O grupo deve interpretar os resultados em uma célula Markdown logo abaixo.

def resumo_inicial(df: pd.DataFrame) -> pd.DataFrame:
    resumo = pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "n_nulos": df.isna().sum(),
        "perc_nulos": (df.isna().mean() * 100).round(2),
        "n_unicos": df.nunique(dropna=True)
    })

    return resumo.sort_values(by="perc_nulos", ascending=False)

# Exemplo de uso:
resumo = resumo_inicial(df)
display(resumo)


,tipo,n_nulos,perc_nulos,n_unicos
age,int64,0,0.0,73
workclass,object,0,0.0,9
fnlwgt,int64,0,0.0,21648
education,object,0,0.0,16
education.num,int64,0,0.0,16
marital.status,object,0,0.0,7
occupation,object,0,0.0,15
relationship,object,0,0.0,6
race,object,0,0.0,5
sex,object,0,0.0,2


In [5]:
# Estatísticas descritivas
# O objetivo é separar análise de variáveis numéricas e categóricas.

def estatisticas_descritivas(df: pd.DataFrame):
    numericas = df.select_dtypes(include=np.number)
    categoricas = df.select_dtypes(exclude=np.number)

    print("Colunas numéricas:", list(numericas.columns))
    print("Colunas não numéricas:", list(categoricas.columns))

    if len(numericas.columns) > 0:
        print("\nEstatísticas descritivas — variáveis numéricas")
        display(numericas.describe().T)

    if len(categoricas.columns) > 0:
        print("\nEstatísticas descritivas — variáveis categóricas/textuais")
        display(categoricas.describe().T)

# Exemplo de uso:
estatisticas_descritivas(df)


Colunas numéricas: ['age', 'fnlwgt', 'education.num', 'capital.gain', 'capital.loss', 'hours.per.week']
Colunas não numéricas: ['workclass', 'education', 'marital.status', 'occupation', 'relationship', 'race', 'sex', 'native.country', 'income']

Estatísticas descritivas — variáveis numéricas


,count,mean,std,min,25%,50%,75%,max
age,32561.0,38.581647,13.640433,17.0,28.0,37.0,48.0,90.0
fnlwgt,32561.0,189778.366512,105549.977697,12285.0,117827.0,178356.0,237051.0,1484705.0
education.num,32561.0,10.080679,2.572720,1.0,9.0,10.0,12.0,16.0
capital.gain,32561.0,1077.648844,7385.292085,0.0,0.0,0.0,0.0,99999.0
capital.loss,32561.0,87.303830,402.960219,0.0,0.0,0.0,0.0,4356.0
hours.per.week,32561.0,40.437456,12.347429,1.0,40.0,40.0,45.0,99.0



Estatísticas descritivas — variáveis categóricas/textuais


,count,unique,top,freq
workclass,32561,9,Private,22696
education,32561,16,HS-grad,10501
marital.status,32561,7,Married-civ-spouse,14976
occupation,32561,15,Prof-specialty,4140
relationship,32561,6,Husband,13193
race,32561,5,White,27816
sex,32561,2,Male,21790
native.country,32561,42,United-States,29170
income,32561,2,<=50K,24720


## 2.1 Exploração inicial — interpretação

[INSERIR ANÁLISE DA EXPLORAÇÃO INICIAL]

A análise deve comentar, no mínimo:

- quantidade de linhas e colunas;
- tipos de atributos;
- valores ausentes;
- variáveis com muitos valores distintos;
- possíveis identificadores;
- possíveis atributos sensíveis;
- variáveis relevantes para a tarefa de mineração;
- limitações iniciais percebidas.

### Relação com as diretrizes escolhidas

[EXPLICAR COMO A EXPLORAÇÃO INICIAL SE RELACIONA ÀS DIRETRIZES]

Exemplos:

- Se a diretriz for **Privacidade e Segurança**, verificar atributos identificadores ou sensíveis.
- Se a diretriz for **Justiça e Viés**, verificar grupos sub-representados ou classes desbalanceadas.
- Se a diretriz for **Eficiência e Escalabilidade**, verificar dimensionalidade, cardinalidade e volume dos dados.
- Se a diretriz for **Explicabilidade**, verificar se os atributos são interpretáveis.


In [6]:
# Visualizações exploratórias básicas
# O grupo deve adaptar as colunas às características do dataset.

def plot_distribuicao_numerica(df: pd.DataFrame, coluna: str) -> None:
    plt.figure(figsize=(8, 4))
    df[coluna].dropna().hist(bins=30)
    plt.title(f"Distribuição de {coluna}")
    plt.xlabel(coluna)
    plt.ylabel("Frequência")
    plt.tight_layout()
    plt.show()

def plot_contagem_categorica(df: pd.DataFrame, coluna: str, top_n: int = 20) -> None:
    plt.figure(figsize=(10, 4))
    df[coluna].value_counts(dropna=False).head(top_n).plot(kind="bar")
    plt.title(f"Contagem de valores — {coluna}")
    plt.xlabel(coluna)
    plt.ylabel("Frequência")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

def plot_matriz_correlacao(df: pd.DataFrame) -> None:
    numericas = df.select_dtypes(include=np.number)

    if numericas.shape[1] < 2:
        print("Não há pelo menos duas variáveis numéricas para calcular correlação.")
        return

    corr = numericas.corr()

    plt.figure(figsize=(8, 6))
    plt.imshow(corr, aspect="auto")
    plt.colorbar()
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
    plt.yticks(range(len(corr.columns)), corr.columns)
    plt.title("Matriz de correlação — variáveis numéricas")
    plt.tight_layout()
    plt.show()

# Exemplos de uso:
# plot_distribuicao_numerica(df, "[COLUNA_NUMERICA]")
# plot_contagem_categorica(df, "[COLUNA_CATEGORICA]")
# plot_matriz_correlacao(df)


## 2.2 Análise visual — interpretação

[INSERIR INTERPRETAÇÃO DOS GRÁFICOS]

A interpretação deve evitar apenas descrever o óbvio. Procure responder:

- Há concentração de valores?
- Há valores extremos?
- Há categorias muito raras?
- Há desbalanceamento?
- Os padrões observados afetam a tarefa de mineração?
- Alguma visualização sugere risco associado às diretrizes escolhidas?

> Não invente conclusões. Toda afirmação empírica deve estar apoiada em uma tabela, gráfico ou cálculo executado no notebook.


In [7]:
# Detecção simples de outliers em variáveis numéricas usando IQR
# Esta função é um exemplo. O grupo deve avaliar se o critério faz sentido para cada atributo.

def detectar_outliers_iqr(df: pd.DataFrame, coluna: str) -> pd.DataFrame:
    serie = df[coluna].dropna()

    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    outliers = df[(df[coluna] < limite_inferior) | (df[coluna] > limite_superior)]

    print(f"Coluna: {coluna}")
    print(f"Q1: {q1}")
    print(f"Q3: {q3}")
    print(f"IQR: {iqr}")
    print(f"Limite inferior: {limite_inferior}")
    print(f"Limite superior: {limite_superior}")
    print(f"Número de outliers: {len(outliers)}")

    return outliers

# Exemplo de uso:
# outliers = detectar_outliers_iqr(df, "[COLUNA_NUMERICA]")
# display(outliers.head())


## 2.3 Problemas de qualidade dos dados

| Problema identificado | Evidência | Decisão tomada | Justificativa |
|---|---|---|---|
| Valores ausentes em [COLUNA] | [TABELA/GRÁFICO/CÁLCULO] | [REMOVER/IMPUTAR/MANTER] | [JUSTIFICAR] |
| Duplicatas | [EVIDÊNCIA] | [DECISÃO] | [JUSTIFICAR] |
| Outliers em [COLUNA] | [EVIDÊNCIA] | [DECISÃO] | [JUSTIFICAR] |
| Categorias raras | [EVIDÊNCIA] | [DECISÃO] | [JUSTIFICAR] |
| Atributos sensíveis ou identificadores | [EVIDÊNCIA] | [DECISÃO] | [JUSTIFICAR] |

### Observação sobre responsabilidade

[EXPLICAR SE ALGUMA DECISÃO DE PREPARAÇÃO TEM IMPLICAÇÕES PARA AS DIRETRIZES ESCOLHIDAS]

Exemplo:

> A remoção de um atributo sensível pode reduzir risco de exposição indevida, mas também pode dificultar a análise de disparidades entre grupos. Essa decisão deve ser justificada com cuidado.


In [8]:
# Exemplo de preparação dos dados
# Esta célula é propositalmente genérica.
# O grupo deve adaptar de acordo com o dataset e a tarefa.

def preparar_dados_basico(
    df: pd.DataFrame,
    colunas_remover=None,
    imputar_numericas=True,
    imputar_categoricas=True
) -> pd.DataFrame:
    df_prep = df.copy()

    # 1. Remover duplicatas exatas
    df_prep = df_prep.drop_duplicates()

    # 2. Remover colunas explicitamente marcadas como irrelevantes ou identificadoras
    if colunas_remover is None:
        colunas_remover = [
            # "[INSERIR_COLUNA_IDENTIFICADORA]",
            # "[INSERIR_COLUNA_IRRELEVANTE]",
        ]

    colunas_existentes = [col for col in colunas_remover if col in df_prep.columns]
    df_prep = df_prep.drop(columns=colunas_existentes)

    # 3. Exemplo de tratamento simples de nulos
    # Atenção: imputação deve ser justificada no texto.
    if imputar_numericas:
        for col in df_prep.select_dtypes(include=np.number).columns:
            df_prep[col] = df_prep[col].fillna(df_prep[col].median())

    if imputar_categoricas:
        for col in df_prep.select_dtypes(include=["object", "category"]).columns:
            df_prep[col] = df_prep[col].fillna("desconhecido")

    return df_prep

# Exemplo de uso:
# df_prep = preparar_dados_basico(df, colunas_remover=["id"])
# display(df_prep.head())


## 2.4 Justificativa da preparação dos dados

[INSERIR JUSTIFICATIVA DAS TRANSFORMAÇÕES REALIZADAS]

Explique:

- quais colunas foram removidas e por quê;
- como valores ausentes foram tratados;
- como variáveis categóricas foram tratadas;
- se houve normalização, discretização ou codificação;
- se houve remoção de outliers;
- quais riscos essas decisões introduzem;
- como essas decisões se relacionam às diretrizes escolhidas.

> A preparação dos dados não deve ser apenas uma sequência de comandos. Toda decisão relevante precisa de justificativa técnica.


## 2.5 Interações com LLM — Data Understanding & Data Preparation

### Subtarefa comparável

Nesta etapa, as duas trilhas devem receber a mesma subtarefa geral.

**Subtarefa escolhida:** [EXEMPLO: propor estratégias de limpeza, transformação e preparação dos dados para a tarefa de mineração]

---

### Trilha A — baseline

**Modelo utilizado:** Gemini Pro
**Data de acesso:** 06/07/2026
**Link da conversa:** (https://gemini.google.com/share/8c72f57d7c9e)

#### Prompt principal — baseline

```text
Tenho o dataset Adult Census Income com as seguintes colunas: age, workclass, fnlwgt, education, education-num, marital-status, occupation, relationship, race, sex, capital-gain, capital-loss, hours-per-week, native-country e a variável-alvo income.
A tarefa de mineração é construir um classificador binário com XGBoost.
Sugira uma estratégia de Data Understanding e Data Preparation para preparar os dados para essa tarefa.
```

#### Síntese da resposta da LLM

Resumo do Pipeline de Preparação (Checklist)

    Substituir ? por NaN.

    Remover colunas fnlwgt e education.

    Binariar native-country (US vs Other).

    Aplicar Target/One-Hot Encoding (ou usar enable_categorical=True).

    Mapear a target income para 0 (≤50K) e 1 (>50K).

    Dividir em treino e teste.

    Calcular o valor de scale_pos_weight com base no treino para passar ao XGBoost.

---

### Trilha B — guiada por diretrizes

**Modelo utilizado:** Gemini Pro
**Data de acesso:** 07/06/2026
**Link da conversa:** https://gemini.google.com/share/2f4b53d8f401

#### Prompt principal — guiado

```text
Tenho o dataset Adult Census Income com as seguintes colunas: age, workclass, fnlwgt, education, education-num, marital-status, occupation, relationship, race, sex, capital-gain, capital-loss, hours-per-week, native-country e a variável-alvo income.
A tarefa de mineração é construir um classificador binário com XGBoost.
Sugira uma estratégia de Data Understanding e Data Preparation.

Considere explicitamente as diretrizes de IA Responsável:
1. Justiça e Mitigação de Viés
2. Explicabilidade

Ao sugerir a preparação, indique:
- possíveis atributos sensíveis, se houver;
- riscos de viés ou perda de informação;
- decisões que devem ser documentadas;
- transformações necessárias;
- limitações e trade-offs.
```

#### Síntese da resposta da LLM

## 2.1. Entendimento dos Dados (Foco em Ética e Viés)
A Análise Exploratória focou na interseção entre a variável-alvo (`income`) e os atributos demográficos. Observa-se:
* **Desbalanceamento Demográfico:** A classe `>50K` é desproporcionalmente ocupada por indivíduos do sexo masculino e de raça branca.
* **Valores Ausentes Não-Aleatórios:** Presença de `?` nas colunas `workclass`, `occupation` e `native-country`, que podem estar correlacionados com classes socioeconômicas mais vulneráveis.

## 2.2. Preparação dos Dados
A transformação dos dados foi desenhada para otimizar a performance do **XGBoost**, garantindo simultaneamente a **Explicabilidade (SHAP)**.

* **Atributos Sensíveis:** `sex`, `race` e `native-country`.
* **Transformações Aplicadas:**
    * **Valores Ausentes:** Conversão de `?` para a categoria explícita `Unknown`, evitando distorções ao imputar a "moda" sobre grupos minoritários.
    * **Variáveis Redundantes:** Remoção da coluna `education` (texto), mantendo apenas `education-num` (numérica e ordinal).
    * **Encoding:** Uso de *One-Hot Encoding* para variáveis nominais (`workclass`, `occupation`, `marital-status`), facilitando a auditoria da importância de cada categoria individualmente.
* **Riscos e Trade-offs:**
    * **Variáveis Proxy:** Remoção da coluna `relationship`, que atua como um forte *proxy* (variável substituta) para o gênero (`sex`), evitando o mascaramento de vieses algorítmicos.
    * **Justiça vs. Acurácia:** O ajuste para mitigação de viés histórico implica em um trade-off consciente, aceitando uma potencial redução em métricas puras de performance global em prol da paridade de tratamento.
* **Decisões Documentadas:** Todas as transformações priorizaram a interpretabilidade das características. Engenharia de atributos complexa (ex: polinômios) foi descartada para garantir que o peso de variáveis legítimas (como `capital-gain` e `hours-per-week`) permaneça claro e explicável.


## 2.6 Comparação crítica — Data Understanding & Preparation

| Aspecto | Trilha A — baseline | Trilha B — guiada | Decisão do grupo |
|---|---|---|---|
| Tratamento de nulos | [SUGESTÃO] | [SUGESTÃO] | [ACEITAR/REJEITAR/CORRIGIR] |
| Tratamento de outliers | [SUGESTÃO] | [SUGESTÃO] | [ACEITAR/REJEITAR/CORRIGIR] |
| Atributos sensíveis | [SUGESTÃO] | [SUGESTÃO] | [ACEITAR/REJEITAR/CORRIGIR] |
| Codificação de variáveis | [SUGESTÃO] | [SUGESTÃO] | [ACEITAR/REJEITAR/CORRIGIR] |
| Redução de dimensionalidade | [SUGESTÃO] | [SUGESTÃO] | [ACEITAR/REJEITAR/CORRIGIR] |
| Justificativa técnica | [ANALISAR] | [ANALISAR] | [DECISÃO] |
| Riscos e limitações | [ANALISAR] | [ANALISAR] | [DECISÃO] |

### Análise crítica

[INSERIR ANÁLISE COMPARATIVA]

A análise deve deixar claro:

- o que a LLM sugeriu corretamente;
- o que estava incompleto;
- o que estava tecnicamente errado;
- o que foi aceito;
- o que foi rejeitado;
- o que foi corrigido pelo grupo.


# 3. Modeling

Nesta seção, o grupo deve documentar a modelagem proposta com apoio da LLM.

Para o TP de padrões frequentes, a modelagem pode envolver, por exemplo:

- Apriori;
- FP-Growth;
- Eclat;
- regras de associação.

Para outros TPs, substituir pelos algoritmos pertinentes, como algoritmos de agrupamento ou classificação.

O foco da Fase 2 é comparar criticamente as soluções sugeridas pela LLM nas duas trilhas e, quando possível, executar testes preliminares para verificar a viabilidade das sugestões.


## 3.1 Interações com LLM — Modeling

### Subtarefa comparável

**Subtarefa escolhida:** [EXEMPLO: propor algoritmos, parâmetros, métricas e estratégia de avaliação para a tarefa]

---

### Trilha A — baseline

**Modelo utilizado:** Gemini Pro
**Data de acesso:** 06/07/2026
**Link da conversa:** (https://gemini.google.com/share/8c72f57d7c9e)

#### Prompt principal — baseline

```text
Com base no dataset Adult Census Income e na tarefa de classificação binária da renda (income), proponha uma estratégia de modelagem focada no algoritmo XGBoost.
Indique parâmetros iniciais, métricas de avaliação e como validar e comparar os resultados.
```

#### Síntese da resposta da LLM

# 1. Business Understanding

* **Objetivo do Dataset:** Prever se a renda anual de um indivíduo excede US$ 50.000 com base em dados demográficos e trabalhistas.
* **Origem:** Extraído por Ronny Kohavi e Barry Becker do banco de dados do *U.S. Census Bureau* (Censo de 1994).
* **Características Gerais:** Possui 32.561 instâncias e 14 atributos (mistura de variáveis contínuas e categóricas). Apresenta um desbalanceamento de classes de aproximadamente 3:1 (76% ganham $\le$ 50K e 24% ganham > 50K).
* **Relação com o Problema:** A classificação auxilia na segmentação socioeconômica para concessão de crédito, marketing direcionado e formulação de políticas públicas.

---

# 2. Data Understanding & Preparation

### Limpeza e Seleção de Atributos
* **Dados Ausentes (`?`):** Substituir por `NaN` para que o XGBoost trate as omissões de forma nativa.
* **Redundância:** Remover a coluna textual `education` e manter a numérica `education-num`.
* **Alta Cardinalidade:** Agrupar a variável `native-country` em `United-States` e `Other`.
* **Irrelevância:** Descartar a coluna `fnlwgt`, pois representa pesos populacionais que podem induzir ao *overfitting*.

### Codificação (Encoding)
* **Variáveis Categóricas:** Aplicar *One-Hot Encoding* ou utilizar o suporte nativo do XGBoost (`enable_categorical=True`).
* **Target:** Mapear a variável `income` para `0` ($\le$ 50K) e `1` (> 50K).

---

# 3. Modelagem com XGBoost

### Estratégia de Validação
* **Divisão dos Dados:** Split inicial em Treino e Teste (ex: 80/20).
* **Validação Cruzada:** Aplicar *Stratified K-Fold* (K=5) na base de treino para garantir a proporção das classes em todos os folds.

### Hiperparâmetros Iniciais (Baseline)
* `objective`: `'binary:logistic'` (classificação binária probabilística)
* `eval_metric`: `'auc'` (foco na separação das classes)
* `scale_pos_weight`: `3.1` (ajuste para mitigar o desbalanceamento)
* `max_depth`: `5` (controle de complexidade inicial)
* `learning_rate`: `0.1` e `early_stopping_rounds=50` (evitar *overfitting*)

---

# 4. Métricas de Avaliação

A performance será avaliada desconsiderando a Acurácia Geral devido ao desbalanceamento. As métricas foco serão:
1. **Matriz de Confusão:** Monitoramento de Falsos Positivos e Falsos Negativos.
2. **F1-Score (Classe >50K):** Equilíbrio entre Precisão e Recall na classe minoritária.
3. **Recall (Classe >50K):** Capacidade de identificar corretamente os indivíduos de alta renda.
4. **AUC-ROC:** Métrica global de capacidade de discriminação do modelo.

---

### Trilha B — guiada por diretrizes

**Modelo utilizado:** Gemini Pro
**Data de acesso:** 07/06/2026
**Link da conversa:** https://gemini.google.com/share/2f4b53d8f401

#### Prompt principal — guiado

```text
Com base no dataset Adult Census Income e na tarefa de classificação binária da renda (income), proponha uma estratégia de modelagem focada no algoritmo XGBoost.

Considere explicitamente as diretrizes de IA Responsável:
1. Justiça e Mitigação de Viés
2. Explicabilidade

Na resposta, indique:
- algoritmos e configurações adequadas;
- parâmetros iniciais;
- métricas de avaliação;
- riscos metodológicos;
- impacto das diretrizes nas escolhas;
- trade-offs entre desempenho, interpretabilidade, custo computacional e qualidade da solução.
```

#### Síntese da resposta da LLM

## 3.1. Algoritmo e Configurações
O modelo escolhido é o **XGBClassifier**. Para garantir alinhamento com as diretrizes éticas, utilizaremos recursos específicos:
* **Monotonic Constraints:** Imposição de restrições lógicas (ex: `capital-gain` e `education-num` só podem ter impacto positivo ou neutro), evitando que o modelo aprenda correlações espúrias.
* **Sample Weights (In-processing):** Uso de pesos compensatórios para instâncias da classe minoritária e de grupos sub-representados para mitigar o viés histórico.

## 3.2. Parâmetros Iniciais (Foco em Explicabilidade)
* `max_depth`: **3 a 5** (Árvores rasas evitam interações ocultas de alta complexidade, facilitando a extração de regras claras via SHAP).
* `learning_rate`: **0.05 a 0.1**.
* `n_estimators`: **100 a 300** (com *Early Stopping* para evitar overfitting).
* `scale_pos_weight`: Calculado ativamente para lidar com o desbalanceamento da variável `income`.

## 3.3. Métricas de Avaliação
O modelo não será validado por acurácia simples, mas por dois eixos independentes:
1. **Métricas de Performance:** ROC-AUC e F1-Score (Macro) para lidar com as classes desbalanceadas.
2. **Métricas de Justiça (Fairness):** * *Equal Opportunity Difference* (Diferença na Taxa de Verdadeiros Positivos entre grupos demográficos).
   * *Disparate Impact* (Razão de aprovação entre grupos).

## 3.4. Riscos e Trade-offs Assumidos
* **Trade-off Desempenho vs. Explicabilidade:** O uso de árvores mais rasas e restrições monotônicas sacrifica marginalmente o poder de captura de não-linearidades em prol de uma auditoria matemática transparente.
* **Trade-off Desempenho vs. Justiça:** A otimização para equidade (ex: alteração do limiar de decisão de 0.5 para subgrupos específicos) reduzirá métricas de performance globais no *Holdout set*, priorizando a paridade de tratamento.
* **Auditoria Contínua:** O risco de variáveis *proxy* atuarem nas decisões será ativamente monitorado com a ferramenta **SHAP (SHapley Additive exPlanations)**, sendo condição de veto para a aceitação do modelo caso atributos de proteção dominem os ganhos de informação.


## 3.2 Comparação das soluções sugeridas pela LLM

| Elemento | Trilha A — baseline | Trilha B — guiada | Diferença concreta? | Comentário do grupo |
|---|---|---|---|---|
| Algoritmos sugeridos | [PREENCHER] | [PREENCHER] | [SIM/NÃO] | [ANALISAR] |
| Parâmetros sugeridos | [PREENCHER] | [PREENCHER] | [SIM/NÃO] | [ANALISAR] |
| Métricas sugeridas | [PREENCHER] | [PREENCHER] | [SIM/NÃO] | [ANALISAR] |
| Preparação exigida | [PREENCHER] | [PREENCHER] | [SIM/NÃO] | [ANALISAR] |
| Interpretabilidade | [PREENCHER] | [PREENCHER] | [SIM/NÃO] | [ANALISAR] |
| Custo computacional | [PREENCHER] | [PREENCHER] | [SIM/NÃO] | [ANALISAR] |
| Riscos ou limitações | [PREENCHER] | [PREENCHER] | [SIM/NÃO] | [ANALISAR] |
| Aderência às diretrizes | [PREENCHER] | [PREENCHER] | [SIM/NÃO] | [ANALISAR] |

### Síntese

[INSERIR ANÁLISE COMPARATIVA]

A comparação deve destacar evidências, não impressões vagas.


## 3.5 Template de modelagem para classificação

Use esta subseção se o TP atual envolver classificação. Caso contrário, remova ou deixe claro que ela não foi utilizada.

> **Atenção:** em classificação, é importante discutir divisão treino/teste, validação, desbalanceamento de classes e escolha de métricas. Acurácia isolada pode ser inadequada em bases desbalanceadas.


In [9]:
# Exemplo genérico para classificação
# O grupo deve adaptar TARGET_COLUMN, tratamento de categóricas e métricas.

def executar_classificacao_basica(df: pd.DataFrame, target_col: str):
    try:
        from sklearn.model_selection import train_test_split
        from sklearn.pipeline import Pipeline
        from sklearn.compose import ColumnTransformer
        from sklearn.preprocessing import OneHotEncoder, StandardScaler
        from sklearn.impute import SimpleImputer
        from sklearn.ensemble import RandomForestClassifier
        from sklearn.metrics import classification_report, confusion_matrix
    except ImportError:
        print("Biblioteca scikit-learn não instalada.")
        return None

    if target_col not in df.columns:
        raise ValueError(f"Coluna-alvo não encontrada: {target_col}")

    X = df.drop(columns=[target_col])
    y = df[target_col]

    colunas_numericas = X.select_dtypes(include=np.number).columns.tolist()
    colunas_categoricas = X.select_dtypes(exclude=np.number).columns.tolist()

    preprocessador = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), colunas_numericas),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]), colunas_categoricas)
        ]
    )

    modelo = Pipeline([
        ("preprocessamento", preprocessador),
        ("classificador", RandomForestClassifier(
            n_estimators=100,
            random_state=RANDOM_STATE,
            class_weight="balanced"
        ))
    ])

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=y if y.nunique() > 1 else None
    )

    inicio = time.time()
    modelo.fit(X_train, y_train)
    tempo_treino = time.time() - inicio

    y_pred = modelo.predict(X_test)

    print(f"Tempo de treino: {tempo_treino:.4f} segundos")
    print("\nRelatório de classificação:")
    print(classification_report(y_test, y_pred))

    print("\nMatriz de confusão:")
    print(confusion_matrix(y_test, y_pred))

    return {
        "modelo": modelo,
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "y_pred": y_pred,
        "tempo_treino": tempo_treino
    }

# Exemplo de uso:
# resultados_classificacao = executar_classificacao_basica(df_prep, TARGET_COLUMN)


## 3.6 Resultados preliminares da modelagem

[INSERIR RESULTADOS OBTIDOS APÓS EXECUÇÃO DO CÓDIGO]

Inclua tabelas e comentários sobre os resultados relevantes para a tarefa:

### Se a tarefa for padrões frequentes

- número de padrões encontrados;
- número de regras geradas;
- suporte, confiança e lift;
- tempo de execução;
- redundância e interpretabilidade das regras.

### Se a tarefa for agrupamento

- número de clusters;
- métricas internas, como Silhouette e Davies-Bouldin;
- tamanho dos grupos;
- interpretação dos clusters;
- estabilidade com diferentes parâmetros.

### Se a tarefa for classificação

- métricas como acurácia, precisão, revocação, F1-score e matriz de confusão;
- análise de classes minoritárias;
- overfitting ou underfitting;
- interpretabilidade e possíveis vieses.

### Atenção

Não conclua que um algoritmo é “melhor” apenas por uma métrica isolada.

Uma comparação adequada deve considerar:

- qualidade da solução;
- interpretabilidade;
- custo computacional;
- estabilidade;
- adequação ao problema;
- relação com as diretrizes escolhidas.


## 3.7 Análise da aderência às diretrizes

| Diretriz | Evidência na Trilha A | Evidência na Trilha B | A Trilha B melhorou? | Justificativa |
|---|---|---|---|---|
| Justiça e Mitigação de Viés | [PREENCHER] | [PREENCHER] | [SIM/NÃO/PARCIALMENTE] | [JUSTIFICAR] |
| Explicabilidade | [PREENCHER] | [PREENCHER] | [SIM/NÃO/PARCIALMENTE] | [JUSTIFICAR] |

### Discussão

[INSERIR DISCUSSÃO]

A resposta deve ser específica. Exemplos de boa análise:

- “A Trilha B sugeriu registrar parâmetros e versões dos experimentos, o que melhora rastreabilidade.”
- “A Trilha B sugeriu avaliar desempenho por subgrupos, o que é pertinente à diretriz de justiça e viés.”
- “A Trilha B recomendou um algoritmo mais interpretável, mas com possível perda de desempenho.”
- “Apesar de mencionar privacidade, a Trilha B não propôs nenhuma alteração concreta no pipeline.”

Evite frases vagas como:

- “A resposta guiada foi mais responsável.”
- “A LLM considerou melhor as diretrizes.”
- “O resultado foi mais ético.”


# 4. Evaluation

Nesta seção, o grupo deve avaliar criticamente:

1. os resultados preliminares obtidos;
2. a utilidade das sugestões da LLM;
3. os erros e limitações das respostas;
4. as diferenças entre a Trilha A e a Trilha B;
5. os trade-offs observados;
6. a contribuição real das diretrizes de IA responsável.


## 4.1 Análise dos resultados

[INSERIR ANÁLISE DOS RESULTADOS]

A análise deve responder:

- Os padrões, clusters, classificações ou resultados encontrados fazem sentido?
- Eles são relevantes para o problema de negócio?
- Há resultados triviais, redundantes ou pouco úteis?
- O desempenho obtido é adequado?
- As métricas usadas são compatíveis com a tarefa?
- Há limitações nos dados ou na modelagem que afetam a conclusão?

> Se os experimentos não produziram bons resultados, isso também é um resultado válido, desde que seja analisado tecnicamente.


## 4.2 Avaliação das sugestões da LLM

| Sugestão da LLM | Trilha | Decisão do grupo | Justificativa |
|---|---|---|---|
| [SUGESTÃO 1] | Baseline | [ACEITA/REJEITADA/CORRIGIDA] | [JUSTIFICAR] |
| [SUGESTÃO 2] | Guiada | [ACEITA/REJEITADA/CORRIGIDA] | [JUSTIFICAR] |
| [SUGESTÃO 3] | Guiada | [ACEITA/REJEITADA/CORRIGIDA] | [JUSTIFICAR] |

### Erros, omissões ou alucinações identificadas

| Problema | Trilha | Por que é um problema? | Como o grupo corrigiu? |
|---|---|---|---|
| [ERRO/OMISSÃO] | [A/B] | [EXPLICAR] | [EXPLICAR] |
| [ERRO/OMISSÃO] | [A/B] | [EXPLICAR] | [EXPLICAR] |

### Comentário crítico

[INSERIR COMENTÁRIO SOBRE A QUALIDADE DAS RESPOSTAS DA LLM]


## 4.3 Comparação final entre as trilhas

| Critério | Trilha A — baseline | Trilha B — guiada | Avaliação crítica |
|---|---|---|---|
| Utilidade prática | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Correção técnica | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Clareza das justificativas | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Aderência às diretrizes | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Qualidade da modelagem | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Qualidade da avaliação | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Riscos não tratados | [ANALISAR] | [ANALISAR] | [COMPARAR] |

### Conclusão comparativa

[INSERIR CONCLUSÃO]

A conclusão deve indicar se a explicitação das diretrizes:

- produziu mudanças concretas;
- produziu apenas mudanças superficiais;
- não produziu diferença relevante;
- introduziu novos trade-offs;
- ajudou a identificar limitações ou riscos;
- levou o grupo a alterar decisões técnicas.


## 4.4 Trade-offs identificados

| Trade-off | Evidência | Decisão do grupo |
|---|---|---|
| Interpretabilidade × desempenho | [INSERIR EVIDÊNCIA] | [INSERIR DECISÃO] |
| Privacidade × utilidade dos dados | [INSERIR EVIDÊNCIA] | [INSERIR DECISÃO] |
| Custo computacional × qualidade da solução | [INSERIR EVIDÊNCIA] | [INSERIR DECISÃO] |
| Simplicidade × completude da análise | [INSERIR EVIDÊNCIA] | [INSERIR DECISÃO] |
| Justiça/viés × disponibilidade de atributos | [INSERIR EVIDÊNCIA] | [INSERIR DECISÃO] |

### Discussão

[INSERIR DISCUSSÃO SOBRE OS TRADE-OFFS]

Nem todo trade-off estará presente em todo trabalho. O grupo deve discutir apenas os que forem pertinentes ao dataset e à tarefa.


## 4.5 Considerações finais da Fase 2

[INSERIR CONSIDERAÇÕES FINAIS]

A conclusão da Fase 2 deve responder:

1. A LLM ajudou em quais partes do trabalho?
2. Em quais partes a LLM errou ou foi superficial?
3. A Trilha B foi de fato diferente da Trilha A?
4. As diretrizes escolhidas tiveram efeito concreto?
5. Quais sugestões serão levadas para a Fase 3?
6. Quais sugestões serão descartadas?
7. O que ainda precisa ser implementado ou corrigido na versão final?

### Próximos passos para a Fase 3

- [ ] Corrigir limitações identificadas nas respostas da LLM.
- [ ] Consolidar a preparação dos dados.
- [ ] Executar modelagem final.
- [ ] Avaliar resultados de forma mais completa.
- [ ] Integrar análise comparativa baseline × guiada × solução final.
- [ ] Revisar documentação e reprodutibilidade.


# 5. Checklist final da Fase 2

Antes de entregar, verifique se o notebook contém:

- [ ] identificação do grupo;
- [ ] link público do dataset;
- [ ] comprovação de que o dataset atende às restrições mínimas;
- [ ] descrição do problema de negócio;
- [ ] descrição das colunas;
- [ ] diretrizes de IA responsável escolhidas;
- [ ] justificativa das diretrizes;
- [ ] hipótese experimental;
- [ ] Trilha A — baseline;
- [ ] Trilha B — guiada;
- [ ] modelo utilizado em cada trilha;
- [ ] data de acesso em cada trilha;
- [ ] links das conversas;
- [ ] prompts principais;
- [ ] síntese crítica das respostas;
- [ ] comparação entre baseline e guiada;
- [ ] sugestões aceitas, rejeitadas e corrigidas;
- [ ] análise de erros e limitações da LLM;
- [ ] conexão com Business Understanding;
- [ ] conexão com Data Understanding & Preparation;
- [ ] conexão com Modeling;
- [ ] conexão com Evaluation;
- [ ] considerações finais e próximos passos.


# 6. Referências
* **WIRTH, R.; HIPP, J.** *CRISP-DM: Towards a standard process model for data mining.* In: Proceedings of the 4th International Conference on the Practical Applications of Knowledge Discovery and Data Mining. London: Springer, 2000.
* **CHEN, T.; GUESTRIN, C.** *XGBoost: A scalable tree boosting system.* In: Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining. ACM, p. 785-794, 2016.
* **XGBOOST DEVELOPERS.** *XGBoost Documentation: Python API Reference.* Disponível em: <\url{[https://xgboost.readthedocs.io/](https://xgboost.readthedocs.io/)}>. Acesso em: 6 de junho de 2026.
* **KOHAVI, R.** *Scaling up the accuracy of naive-bayes classifiers: A decision-tree hybrid.* In: Proceedings of the Second International Conference on Knowledge Discovery and Data Mining (KDD). p. 202-207, 1996.
* **DUA, D.; GRAFF, C.** *UCI Machine Learning Repository: Adult Dataset.* Irvine, CA: University of California, School of Information and Computer Science, 2019. Disponível em: <\url{[https://archive.ics.uci.edu/ml/datasets/adult](https://archive.ics.uci.edu/ml/datasets/adult)}>. Acesso em: 6 de junho de 2026.
* **BAROCAS, S.; SELBST, A. D.** *Big Data's Disparate Impact.* California Law Review, vol. 104, no. 3, p. 671-732, 2016.
* *Uso no projeto:* Fundamentação teórica para debater os conceitos de Impacto Díspar, discriminação por variáveis *proxy* e reprodução automatizada de desigualdades históricas de gênero e raça.
* **MELLON, J.** *The Four-Fifth Rule and Statistical Equity in Algorithmic Decision-Making.* Journal of Machine Learning Ethics, vol. 12, p. 45-58, 2022.
* **LUNDBERG, S. M.; LEE, S.-I.** *A unified approach to interpreting model predictions.* In: Advances in Neural Information Processing Systems (NeurIPS). p. 4765-4774, 2017
* **LUNDBERG, S. M. et al.** *From local explanations to global understanding with explainable AI for trees.* Nature Machine Intelligence, vol. 2, no. 1, p. 56-67, 2020.
* **PEDREGOSA, F. et al.** *Scikit-learn: Machine Learning in Python.* Journal of Machine Learning Research, vol. 12, p. 2825-2830, 2011.
* *Uso no projeto:* Suporte à implementação do pipeline de pré-processamento, validação cruzada estratificada (`StratifiedKFold`) e extração de métricas tradicionais de erro ($F1\text{-Score}$ e matriz de confusão).


# Apêndice A — Modelo de registro de interações com LLM

Esta seção é opcional, mas recomendada para organizar evidências.

| ID | Trilha | Subtarefa | Modelo | Data | Link | Prompt resumido | Decisão do grupo |
|---|---|---|---|---|---|---|---|
| A1 | Baseline | Business Understanding | [MODELO] | [DATA] | [LINK] | [RESUMO] | [ACEITA/REJEITADA/CORRIGIDA] |
| B1 | Guiada | Business Understanding | [MODELO] | [DATA] | [LINK] | [RESUMO] | [ACEITA/REJEITADA/CORRIGIDA] |
| A2 | Baseline | Data Preparation | [MODELO] | [DATA] | [LINK] | [RESUMO] | [ACEITA/REJEITADA/CORRIGIDA] |
| B2 | Guiada | Data Preparation | [MODELO] | [DATA] | [LINK] | [RESUMO] | [ACEITA/REJEITADA/CORRIGIDA] |
| A3 | Baseline | Modeling | [MODELO] | [DATA] | [LINK] | [RESUMO] | [ACEITA/REJEITADA/CORRIGIDA] |
| B3 | Guiada | Modeling | [MODELO] | [DATA] | [LINK] | [RESUMO] | [ACEITA/REJEITADA/CORRIGIDA] |


In [10]:
# Apêndice B — Estrutura opcional para registrar prompts e respostas de forma organizada
# Esta estrutura pode ajudar o grupo a manter rastreabilidade.
# Evite inserir respostas muito longas no notebook; prefira sínteses críticas e links das conversas.

registros_llm = [
    {
        "id": "A1",
        "trilha": "baseline",
        "subtarefa": "Business Understanding",
        "modelo": "[INSERIR MODELO]",
        "data": "[INSERIR DATA]",
        "link_conversa": "[INSERIR LINK DA CONVERSA]",
        "prompt_resumido": "[INSERIR RESUMO DO PROMPT]",
        "sintese_resposta": "[INSERIR SÍNTESE DA RESPOSTA]",
        "decisao_grupo": "[ACEITA/REJEITADA/CORRIGIDA]",
        "comentario_critico": "[INSERIR COMENTÁRIO]"
    },
    {
        "id": "B1",
        "trilha": "guiada",
        "subtarefa": "Business Understanding",
        "modelo": "[INSERIR MODELO]",
        "data": "[INSERIR DATA]",
        "link_conversa": "[INSERIR LINK DA CONVERSA]",
        "prompt_resumido": "[INSERIR RESUMO DO PROMPT COM DIRETRIZES]",
        "sintese_resposta": "[INSERIR SÍNTESE DA RESPOSTA]",
        "decisao_grupo": "[ACEITA/REJEITADA/CORRIGIDA]",
        "comentario_critico": "[INSERIR COMENTÁRIO]"
    }
]

df_registros_llm = pd.DataFrame(registros_llm)
display(df_registros_llm)


,id,trilha,subtarefa,modelo,data,link_conversa,prompt_resumido,sintese_resposta,decisao_grupo,comentario_critico
0,A1,baseline,Business Understanding,[INSERIR MODELO],[INSERIR DATA],[INSERIR LINK DA CONVERSA],[INSERIR RESUMO DO PROMPT],[INSERIR SÍNTESE DA RESPOSTA],[ACEITA/REJEITADA/CORRIGIDA],[INSERIR COMENTÁRIO]
1,B1,guiada,Business Understanding,[INSERIR MODELO],[INSERIR DATA],[INSERIR LINK DA CONVERSA],[INSERIR RESUMO DO PROMPT COM DIRETRIZES],[INSERIR SÍNTESE DA RESPOSTA],[ACEITA/REJEITADA/CORRIGIDA],[INSERIR COMENTÁRIO]
